# IMPORT

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# DEFINE FOLDERS

In [3]:
BASE_FOLDER = Path("../data")
DATASET_INPUT_FOLDER = BASE_FOLDER / "00_raw" / "csv" / "correlated_data_csv"
DATASET_OUTPUT_FOLDER = BASE_FOLDER / "01_clean" / "datasets"
PLOT_FOLDER = BASE_FOLDER / "05_visualization" / "correlation_plots"

folders = [DATASET_INPUT_FOLDER, DATASET_OUTPUT_FOLDER, PLOT_FOLDER]
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

# DEFINE CONSTANTS

In [4]:
REMAKE_CSV = False # To get the dataset from json files(will be saved in a csv file)
CSV_FILE_NAME = "correlated_data" #To save the dataset from json to csv/to get the dataset from csv
FINAL_CSV_FILE_NAME = "final_clean_split_data" # The prepared dataset

# DEFINE FUNCS

In [15]:
def save_station_correlation(
    df: pd.DataFrame,
    station_code: int | None,
    save_path: Path = PLOT_FOLDER / "correlation_matrix.png",
    title_prefix: str = "Feature Correlation Matrix",
    bike_lag: bool = False
):
    """
    1) takes a dataframe and an optional station code
    2) computes the Pearson correlation matrix for selected columns
    3) saves the correlation heatmap to disk
    """

    SELECTED_COLS = [
        "total_bikes_now",
        "total_bikes_taken",
        "temperature",
        "is_day",
        "wind_speed",
        "wind_direction",
        "weather_code",
    ]
    if bike_lag:
        SELECTED_COLS.append("total_bikes_lag_1")
        SELECTED_COLS.append("total_bikes_lag_2")

    # --- numeric selection ---
    numeric_df = df.select_dtypes(include=[np.number]).copy(deep=True)

    # --- optional station filtering ---
    if station_code is not None:
        numeric_df = numeric_df[numeric_df["station_code"] == station_code]

    # --- keep only requested columns ---
    numeric_df = numeric_df[SELECTED_COLS].dropna(how="all")

    # --- correlation ---
    corr_matrix = numeric_df.corr(method="pearson")

    # --- colormap ---
    cmap = LinearSegmentedColormap.from_list(
        "corr_cmap", ["blue", "white", "red"]
    )

    # --- plot ---
    plt.figure(figsize=(11, 9))
    im = plt.imshow(corr_matrix, cmap=cmap, vmin=-1, vmax=1)
    plt.colorbar(im, label="Pearson correlation")

    plt.xticks(
        ticks=range(len(corr_matrix.columns)),
        labels=corr_matrix.columns,
        rotation=45,
        ha="right"
    )
    plt.yticks(
        ticks=range(len(corr_matrix.columns)),
        labels=corr_matrix.columns
    )

    for i in range(corr_matrix.shape[0]):
        for j in range(corr_matrix.shape[1]):
            value = corr_matrix.iloc[i, j]
            plt.text(
                j,
                i,
                f"{value:.2f}",
                ha="center",
                va="center",
                color="black" if abs(value) < 0.5 else "white",
                fontsize=9
            )

    title = title_prefix if station_code is None else f"{title_prefix} - Station {station_code}"
    plt.title(title)
    plt.tight_layout()

    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path)
    plt.close()

# LOAD DATA

In [6]:
df = pd.read_csv(f"{DATASET_INPUT_FOLDER}/{CSV_FILE_NAME}.csv")
df.head(5)

C:\Users\emirh\AppData\Local\Temp\ipykernel_443236\226569353.py:1: DtypeWarning: Columns (2,3,4,5,8,9,14,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{DATASET_INPUT_FOLDER}/{CSV_FILE_NAME}.csv")


,name,stationcode,ebike,mechanical,latitude,longitude,duedate,numbikesavailable,numdocksavailable,capacity,...,is_returning,code_insee_commune,weather_time,weather_interval,weather_temperature_2m,weather_is_day,weather_wind_speed_10m,weather_wind_direction_10m,weather_weather_code,source_file
0,Cassini - Denfert-Rochereau,14111,2,0,48.837526,2.336035,2025-11-04T03:25:04+00:00,2.0,22,25,...,OUI,75056,2025-11-04T04:00,900.0,10.1,0.0,9.5,217.0,3.0,correlated_20251104_051241.json
1,Basilique,32017,5,1,48.936269,2.358867,2025-11-04T03:21:03+00:00,6.0,16,22,...,OUI,93066,2025-11-04T04:00,900.0,10.1,0.0,9.5,217.0,3.0,correlated_20251104_051241.json
2,Charonne - Robert et Sonia Delaunay,11104,1,1,48.855908,2.392571,2025-11-04T03:29:35+00:00,2.0,18,20,...,OUI,75056,2025-11-04T04:00,900.0,10.1,0.0,9.5,217.0,3.0,correlated_20251104_051241.json
3,Lacépède - Monge,5110,1,6,48.843893,2.351966,2025-11-04T03:30:23+00:00,7.0,15,23,...,OUI,75056,2025-11-04T04:00,900.0,10.1,0.0,9.5,217.0,3.0,correlated_20251104_051241.json
4,Saint-Romain - Cherche-Midi,6108,2,2,48.847082,2.321375,2025-11-04T03:28:05+00:00,4.0,11,17,...,OUI,75056,2025-11-04T04:00,900.0,10.1,0.0,9.5,217.0,3.0,correlated_20251104_051241.json


In [7]:
df_clean = pd.read_csv(f"{DATASET_OUTPUT_FOLDER}/{FINAL_CSV_FILE_NAME}.csv")
df_clean.head(5)

,timestamp,station_code,latitude,longitude,capacity,total_bikes_now,total_bikes_taken,temperature,wind_speed,wind_direction,weather_code,is_day,hour,day_of_week,is_weekend,sin_hour,cos_hour,split,total_bikes_lag1,total_bikes_lag2
0,2025-11-04 04:30:00,1001,48.857059,2.341798,17.0,2.0,15.0,9.0,8.0,207.0,3.0,0.0,4,1,0,0.866025,0.500000,0,2.0,2.0
1,2025-11-04 04:45:00,1001,48.857059,2.341798,17.0,2.0,15.0,9.1,8.2,203.0,3.0,0.0,4,1,0,0.866025,0.500000,0,2.0,2.0
2,2025-11-04 05:00:00,1001,48.857059,2.341798,17.0,2.0,15.0,9.1,8.2,203.0,3.0,0.0,5,1,0,0.965926,0.258819,0,2.0,2.0
3,2025-11-04 05:15:00,1001,48.857059,2.341798,17.0,2.0,15.0,9.1,8.1,201.0,3.0,0.0,5,1,0,0.965926,0.258819,0,2.0,2.0
4,2025-11-04 05:30:00,1001,48.857059,2.341798,17.0,2.0,15.0,9.1,8.4,200.0,3.0,0.0,5,1,0,0.965926,0.258819,0,2.0,2.0


# FORCE NUMERIC VALUES TO BE NUMERIC

In [8]:
MISSING_STRINGS = {"", "na", "n/a", "null", "none", "nan", "-"}

# Normalize missing-like strings
df = df.replace(list(MISSING_STRINGS), np.nan)

# Columns that must be numeric
NUMERIC_COLS = [
    "latitude",
    "longitude",
    "capacity",
    "numbikesavailable",
    "numdocksavailable",
    "weather_temperature_2m",
    "weather_is_day",
    "weather_wind_speed_10m",
    "weather_wind_direction_10m",
    "weather_weather_code",
]

for col in NUMERIC_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce") # coerce to set to nan

df.replace([np.inf, -np.inf], np.nan, inplace=True)


# RENAME FEATURES(COLUMNS)

In [9]:
feature_name_dict = {
    "name": "station_name",
    "stationcode": "station_code",
    "numbikesavailable": "total_bikes_now",
    "numdocksavailable": "total_bikes_taken",
    "nom_arrondissement_communes": "commune_name",
    "code_insee_commune": "postal_code",
    "weather_time": "timestamp",
    "weather_temperature_2m": "temperature",
    "weather_is_day": "is_day",
    "weather_wind_speed_10m": "wind_speed",
    "weather_wind_direction_10m": "wind_direction",
    "weather_weather_code": "weather_code"
}

df = df.rename(columns=feature_name_dict)

# CORRELATION MATRIX

In [14]:
save_station_correlation(df, station_code=None, save_path=PLOT_FOLDER / "correlation_matrix.png", title_prefix="Feature Correlation Matrix")
save_station_correlation(df_clean, station_code=None, save_path=PLOT_FOLDER / "correlation_matrix_clean.png", title_prefix="Feature Correlation Matrix (Cleaned)")

KeyboardInterrupt: 

In [19]:
station_codes = [11014, 11022, 2017]

for station_code in station_codes:
    save_station_correlation(df, station_code=station_code, save_path=PLOT_FOLDER / f"station_{station_code}_correlation_matrix.png", title_prefix=f"Feature Correlation Matrix")
    save_station_correlation(df_clean, station_code=station_code, save_path=PLOT_FOLDER / f"station_{station_code}_correlation_matrix_clean.png", title_prefix=f"Feature Correlation Matrix (Cleaned)")
    # save_station_correlation(df_clean, station_code=station_code, save_path=PLOT_FOLDER / f"station_{station_code}_correlation_matrix_clean_bike_lag.png", title_prefix=f"Feature Correlation Matrix (Cleaned) with Bike Lag", bike_lag=True)
